# 🕳️ Lab 1 — The Leakage Hunt

**Mission briefing.** Your teammate built a fraud-detection model and posted this in Slack:

> *"Fraud model is at **0.978 test AUC** 🎉 — creating the deploy ticket, sound good?"*

That number is too good. Real-world fraud models on features like these live around 0.80–0.85.
Your job: **audit the pipeline before it ships.** There are **three planted flaws** that each
inflate the test score. Find all three, fix them, and report the model's *honest* AUC.

**Rules of engagement**
1. Run the 🔒 sealed cell first (it provides the data loader and the grader). **Never decode it** — that's the answer key.
2. Read the teammate's script, interrogate the data, form hypotheses, test them.
3. Submit with `submit_verdict(findings=[...], honest_auc=...)`. You get instant found/missed counts; the full debrief unlocks on a pass.
4. Stuck? `hint(1)`, `hint(2)`, `hint(3)` escalate. Using hints is fine; decoding the blob is not.

**Findings must be ids from this taxonomy** (some are decoys — false alarms are counted):

| id | claim |
|---|---|
| `post-label-feature` | a feature is computed after / because of the label |
| `duplicates-across-split` | the same rows appear on both sides of the split |
| `target-encoding-leak` | an encoding was computed using labels outside the training fold |
| `scaler-leak` | feature scaling was fit on the full dataset |
| `test-set-tuning` | hyperparameters were tuned against the test set |
| `temporal-split-needed` | the data is temporal; a random split is invalid |
| `auc-invalid-under-imbalance` | AUC is a misleading metric at this class balance |
| `label-noise-inflation` | noisy labels are inflating the score |
| `lucky-seed` | the result is random-seed luck |
| `wrong-metric-use-accuracy` | accuracy should have been used instead |

**Pass = all 3 planted flaws claimed + honest AUC inside the sealed key's tolerance.**

In [ ]:
#@title 🔒 SEALED GRADER — run me first. Do NOT decode by hand: it spoils the whole lab.
import base64 as _b64, hashlib as _hl
_blob = (
    "pYT2Qm4iUU0t7uK/Y3e0Boua2KYk3++mggKa833q2r7wnf84Cw92KAbfr/I6ZuZGnM+8sm2Zv5iLA5WodLGjvo/Nu0BvNwVQY8Th"
    "vzQltBvPqtv4bJr5ppkKj98v5c628L7ed0V5L01ju6+hOmr1RJ31huYC37/nzAeW7yjl3b6SzcRccX5AFTOz/aF9ebsa2ajXuiDM"
    "sfLARsqubKeJ8IbEkRIhcAUFLO797yd3pxvM69+4fJr4op4V07Bxq5uqg831GwtwBU1j+uysdSK7AfSk0bMowr+1ggHV5zzmxP+H"
    "37UCLXARXXO3r6EzXfVVi+XGpGGQ7ZiYHpXzfbaJ7MGKtUJuOVYeLPWn/Cp79RuCz5b2KN/5qJ4Dkuczq5S+3YP8HHMxSwks9qeh"
    "M3fpVZvrh+QC37/nzAik7Tj5yvbOg+9BIW0FX3Orv8U6d/VVxqDEtWCe8bOzD5+gYKvb8MjD8lx1NUIIMein/zZ3uyrGoMS1YJ7x"
    "s59K2+50gYm+j832V3MzTQwt79C9cyS+VZblxLhv0f2imAfTsXO5hb6e37cSbw9ICDH45650I6ZcoeWW9iiT8KCFEtu9faOjvo/N"
    "uxIhcAVAcbW6+hB39VWL5Zb2KNS/98JezqB3q4HBwZ21Xm43DQwu9Pqhbn71WIv2mOMh9b/nzEbboH2rgr6ew64SK3BDAjH+5qh0"
    "XfVVi+WW9ijftOfdSMKgd6uB9sCY6RI9cBNESbuv7zp39VWL6JbmJs+v9d5G0aA86Mrx2oPvbWA3QGdju6/vOnf1VYblhvg4za3n"
    "xkaL8jTk28HblfVBC3AFTWO7r+86fPVBhfCW/CiS+rWPDpruKdTb99yGwF9kIkYFIvX7kHMziH+L5Zb2KN+/58dGie46pcfx3YD6"
    "XilgCU1ztbr6Nne7XKHllvYo1pXnzEbb8H22ia+PwrsaMHAOTRz1/+F/L6VdhqnZsWGLtu7mRtugfe3b/9qJuw8heFcDJLX9rnQz"
    "uhiDq5/2NN/v7sIHiPQk+8y2xoPvGwtwBU1j/eOufTCwEYv4lolmj7GwhAOJ5XXt2//aibsPPHAUQWPp4ag0JbQbz6rb/mbWv/vM"
    "VtW3aKeJ7MGKtUBgPkECLrPh5jpr9UWF9YL/At+/58wCnaBgq/buy8PfU3UxYx8i9urnYV31VYvllvYo372zlAik6Tmpk77wg+sc"
    "YCJEAyT+p/42d7tVgOWH/yjVv/DMTduxbbuZrp/dtzghcAVNY7uv7zg2uBreq8L0Mt/+qoMTlfRz+cbrwYmzACh8L01ju6/vOnf1"
    "V8Oqw6Qqxb+vgxOJrFerib6PzbsSIXJEDiD0+qFuCLQSzprSt3GMvf3MB5jjMv7H6vCM/FcvIkoYLf+n/zN731WL5Zb2KN+/5ZwU"
    "ku8v1N3mwZ65CCEgVwQs6dC7YjmmWaHllvYo37/nzESd7y/uwPnBz6ESZz9XCCr84eF7JKEM26Cev2aLtuvmRtugfauJvo/P9ldz"
    "M00MLe/Qpn5171XGoMS1YJ7xs7MPn6xXq4m+j827EiFyQwEi/OiqfgizGtmaxLN+lvqwzlzb5jHqzvnKibVTciRcHSaz5qFufvl/"
    "i+WW9ijfv+fOAInhKO+LpI+L6VN0NAlnY7uv72d+31WL5ZakZpit59FGpO4tpdv/wYn0Xy80QAsi7uO7RSW7EoOa5ZNNu7/szFfS"
    "in2rib7LmOttaDRdTX67/aF9ZfsWw6rftW3X86KCTp/mdKeJ98GZswIvYRVNabvjqnR/sROC7Jr2eprvq40Fnr0b6sXtysSREiFw"
    "BQklu7LvRSexW8iq2LVpi7eciADXoDnth/fDgvhpZSVVMir/95JHe/UczKvZpG2g9qmIA4O9Cfnc+4bnuxIhcFcIN+79oTozs1vY"
    "pNumZJq3oZ4HmL1sp4nszoP/XWwPVhki7+ryRQSQMO/lnfY51rG1iRWe9ALix/rKlbNWcz9VUBfp+qozXd8RzqOWumee+5iYFJru"
    "LurK6saC9UEpeR9nY7uv7zh19ybCqMO6aYv6tNZGqMURzurKj8e7dFMfaE00+v2qcjigBs7rwq5moPmijRKO8jj49uiczbNGaTUF"
    "GSb64qJ7I7BS2OXSt3yev7eZCpepc6mLvKXNuxIhIkAZNunh70UwsBuD7LzcV6/ThqIyvsR9tonlpc27EiFyVQIw76KjezWwGYaj"
    "07d8iu2izlzbqFerib6PzbsSIXJFCy/66Kh/M4oTxLfppG2J9qKbBtvpLqva+9vN2nRVFXdNN/Pq73wltADP5dmjfJzwqolGkvN9"
    "4Mfx2IO70IHEBQQ3u+a8Ojb1B86z37N/3/mrjQHbolerib6PzbsSIXJRBSLvr6J1JKEZ0uXTrmGM67TMBJ7jPP7a+4+Z81chJFcM"
    "LejurG4+uhuLpNqkbZ77vswSjvIz7s2+wJjvEnU/BQ8mu+m9eyKxVYPyg/MokPnnihSa9Tmri5SPzbsSIXAFTWHp4Lhpd7QHzuXQ"
    "ummY+KKIRo3zfb+MvsCLu1FtNUQDY+nguGl++1XqsZamepr7ro8Sku8zq933woi7Rmk5Vk0g9OO6dzn1AsSw2rIokfCzzAOD6S7/"
    "iefKmbUSI1oFTWO7r+86d/c4zqTFo3qa++eJAJ3lPv+Tvs6J/1tvNwUEN7v7oDo2u1XEsd6zeoj2tIlLk+8z7trqj53yQmQ8TAMm"
    "u+ahfDu0Ac62lqJtjOvnrTO4oD/yiezAmPxabSkFRnO1vv40d/d/i+WW9ijfv+fONJ7mMe7RpI+L9EAhNVMIMeKvqX82oQDZoJa3"
    "e5S/4JsJjuw5q+C+x4ztVyEkTQQwu/mudiKwVcqxlqJgmr+qgwue7imrxviPnelXZTlGGSr04fA9d/d/i+WW9ijfv+fONJ72NO7e"
    "pI+E9VZkKAsFN/bj7Gk8vBnH6NqzaZT+oIlE8aB9q4m3g+e7EiFwBwk26+OmeTahENjo17V6kOy0wRWL7DT/i6SPxZESIXAFTWO7"
    "r+1bNboA3+WH5i3f8KHMFJT3LqvI7t+I+kAhJFIEIP6v53t3txrfpt6zbN/qt58SieU85onywIq7WG45S02hGxvvaTa4EIuxzrhX"
    "lvvrzA+f5TP/wP3OgbtEYDxQCDCyoe84XfVVi+WW9ijfvYbMFZP1O+3F+8vN6EJtOVFNM+77vDo4uxCLptmmcd/2qcwSieE05Yn/"
    "wYm7XW81BQQtu/uqaSP5VdiqlqJgmr+qgwKe7H3i2r7In/pWZDQFHSLp+6Njd7obi+e89ijfv+fMRtuiL+Te7Y+E7xJsNUgCMfL1"
    "qn559TjOpMWjepr754kAneU+/5O+3YLuVWk8XE1oq6H/KLVV5pvrhuUovsqEzAmVoCnk2b7Ai7tTb3BNAi3+/Ls6J7wFzqnfuG3R"
    "v+XmRtugfauJvo/PyVdnPEAVebvsp380vlXLodD4bIrvq4UFmvQ474G3gZ7uXyl5RU1r+uGrOjmwFNno0qN4k/akjRKe833kx77b"
    "hf4SZD5RBDfir6R/LvxVyaDQuXqav4aiP9vzLefA6oHNuTghcAVNY7uv7zgFsAPCoMHsKJbxo4ke1egp5sW93IbyXm19Uwwv8uuu"
    "bj66G4nPlvYo37br5kbboH2p3f/div5GLDVLDiz/5qF9erkQyq6U7CjXlefMRtugfauJvM+A/kBiOEQDN8TqoXk39RzY5cK+bd/y"
    "op4Fk+Ez/47tj4vpU3Q0BR8i7+rveTi4Bd6x07IokOminkaP6Dir78vjobtWYCREHibvry2aw/Ucxabao2yW8aDMEpPlfamjvo/N"
    "uxIhcAVPN/78uzolugLY4pa6aZ36q59I28Ur7tvnj5n+QXVwVwI0vPzvdSC7Vcek1LNk3/a0zASa6zjviffBmfQSaCRWTSzs4e98"
    "MrQB3rfT+Cio9rOERoWycbuZro/PkRIhcAVNY7uv7XcypxbDpNiie9/+s8wYw6Av5N7tj4j6UWl8BRkr+vvvcyT1FIuh36RtnOvn"
    "nA+L5X3t2/HCze9XciQFASL56qNpd7wb36qWomCav6qDAp7sc6uLlI/NuxIhcAVNYdbqrmkipxDP5dOwbpr8s9ZGie8o7MHy1s2w"
    "Ai9hFU0CzszhOhG8AYug2LVnm/apixXb7zOr3fbKze9AYDlLBC38r6l1O7FVxKvaryjX8LKYS5TmcO3G8svNuTghcAVNY7uv7zgg"
    "vAHDrNj2fI3+roJK2+k57sjyw5SyHiE/V00n6eC/OiO9EIum2bp9kvHpzCSU7ij4ierdjOsIITEFSjfp7qZ0erobx7yR9m2R/KiI"
    "D5Xnff/B/9vN6EZoPElNYZGv7zp39VWL5ZS/ZpzzsogDiKA46sr2j5npU2g+TAMku/2gbXCmVeSS+PZknv2igEaS7n3i3e2PgP5A"
    "YjhEAze74qp7OfUYyq7TpSiL96LMC5TkOOeJ8dmI6VRoJAUZK/6v7RB39VWL5Zb2KN36qY8Jn+kz7In/wYm7UXMxUQgx6K+7fySh"
    "VeqQ9fZ8kL+53EjOs31pKQqPhP0SeD9QTS/64at/M/UBw6DEsyTf66+NEtvpLqve9tbDuxALcAVNY7uv7zp1hxDdrNOhMt/2qYgD"
    "g641/8TyjJ7wW208CAEm+uSufTL1FMWhlr9mm/q/wg6P7TGoxPLXwPhAbiNWQDX646Z+NqEcxKuU3Cjfv+fFSvH9V4H22uqu1GtS"
    "cBhNOJGv7zp39wbIpNqzetLzoo0N2bp9qf32yp/+EmgjBQMsu/ysezuwB4uk2K9/l/q1iUaS7n3/wffczetbcTVJBC3+oe02XfVV"
    "i+WUom2M6+qfA4+tKf7H98GKuQghcnEFJrv8rGg+pQGLscS3YZHs54MImOV9/MDqx839W3k1QU0r4v+qaCe0B8qo06JtjeznDuZv"
    "oDPk3fbGg/wSdjFWTTfu4ap+d7QSyqzYpXzf66+JRo/lLv+J7cqZtRAtWgVNY7utu386pRrZpNr7e4/zrphLleU478z6jde7EFU4"
    "QB4mu/u9ezmmFMix37lmjL+vjRCeoDPkifHdif5AaD5CQjfy4qppI7QY2+XVuWSK8qnXRpXvKePA8MjN+lBuJVFNN/Pq7242pxLO"
    "sZayepb5s59GlPY4+Ynqx4i7VGg8QENj2q+9ezmxGsblxaZkluvnhRXb4T7ozO7bjPleZHBNCDH+oe02XfVVi+WUt32csq6CEJrs"
    "NO+E68GJ/kAsOUgPIvfuoXky90+L5/eDS9/2tMwUmu42psv/3Ij/EmA+QU0x/uKuczmmVcrlwLdklvvnjwmW8Dz5wO3Ag7tfZCRX"
    "BCC77rs6NvVCjuXGuXuW666aA9vyPP/MsI/Fy2AsEXAuY+zgunYz9TTnlvn2apq/sIMUj+h9+czuwJ/vW283CU0h7vvvWwKWVcmg"
    "37hv37iughCa7DTvjr7GnrtcbiQFAi3+r6B8d6EdzuXGummR66KIRp3sPPzasIbPtzghcAVNYffurX87+BvErMWzJZbxoYAHj+ky"
    "5Yukj8/XU2M1SR5j+v2qOjm6HNi8mvZqiuvngAeZ5TGrx/HGnv4SRRVjIQLPypw6NqUFyrfTuHzf76KeAJTyMOrH/crWu1t1cEYM"
    "LfXguzoyrQXHpN+4KJ6/s4MJ1ucy5M2+3I70QGR+B0FJu6/vOnW5AMiuz/t7mvqjzlzbog/uhOzag7tFaCRNTSzv56pod6YFx6zC"
    "9nua+qOfXNv0Ne6J7MaK/FdlcFYOLOnq72kjtAzY5cjmJsan6cwvj6A0+InwwJm7QWQ1QU0v7uykNHX5f4vllvYqiO2oggHW7Tj/"
    "2/fMwO5BZH1EDiDu/a55LvdPi+f3tWuK7aaPH9v3Mv7F+o+P/hJWH3c+Brvnqmgy9V3K5dK5JZHws4QPled95sb6yoG7VWQkVk09"
    "orzqM3n1NP6Glr973/7nngOa8zLlyPzDiLtRaT9MDia1reMQKt9/9JH3jkex0Iq1RsagMeLa6oeyy35AHnEoB7Kv5Do7vAbf7emS"
    "TbzQnr9P8d8VxOfb/LnEfk58BTIL1MGKSQOKPeLli/Y40af3wEbLrmW9o8HOme9XbCBRHmOmr5QqCt9/z6DQ9mCW8bPECMaxdLGj"
    "vo/Nu1poPlEeY6avlBB39VWL5Zb2KN3XroIS27FyuJO++5/uQXVwSwI38+ahfXn1Jdmq0L9kmr+zhAPb8jz8ierOj/dXITJACyzp"
    "6u9oMrQRwqvR9nyX+ueBCZ/lMavK8cuIu9CBxAUfLOyvrHUiuwHY6ZajZpbusokInvMuq8b4j5njXF45QUFj+uGrOj+6Aoug17Vg"
    "3/yogBOW7n35zPLOmf5BISRKTSP9/a5vM7Vbiem89ijfv+fMRtuiFeLH6o/ftAE7cGMCMbvquX8lrFXNoNeifY3668wHiOt93OHb"
    "4c3yRiEzSgAm6K+mdCO6Vc6936V8mvGkiUaJ5THq3ffZiLtGbnBRBSa76b17IrFVx6TUs2TRv4iCA9vvO6vd9sqe/hJiP0kYLvX8"
    "73Mk9RTF5dOwbpr8s8wJnaAp48y+w4z5V218BQMs76+uOjS0ANigmPQk9b/nzEbboH2ri9bGg+8SMn8WV2P74qpoNL0UxbHps2ac"
    "/+eFFdvjMubZ69uI/xJnIkoAY/vrqXp3N/U/5cK+bd/or4MKnqA57Ye++IXyUWlwVwI06Kjvdja3EMe2lrJnmuznjUavxQ7fiezA"
    "mrxBITVLDiz/5qF9d7IQ3+XCuSiM+qLTRNeKfauJvvLnuxIhcEtNfrvirmJ/5FmLqN+4IMyz54UIj6gzooC3pc27EiEgVwQt76en"
    "czmhBvCrlvsozsLu5myf5Tur2uvNgPJGXiZAHyfy7LsyMbwbz6zYsXvTv6+DCJ7zKdTI68zBu1xuJEAefrmt5iBd9VWL5d+7eJDt"
    "s8wOmvM158D8j4zoEl44L01ju6+QeyOhEMa1wqVTz8Lnx1vbsVerib6Pi/JcZTlLCjC7su92PqYBg6HftXzR+bWDC5DlJPiB+MaD"
    "/1tvN1ZEapGv7zp3oBvAq9mhZt+i57cA2+Yy+Yn4j4T1Emc5Swkq9ei8Oj6zVc3l2Ll83/apzDmvwQXE59HitMY4IXAFTSr9r7p0"
    "PLsa3KuM3Cjfv+fMRtugLfnA8NvF/RBUPk4DLOzh73w+uxHCq9H2YZu3tMVc2/so5cLwwJr1TyN5L01ju6/vOnf1Bdms2KIg3dyv"
    "gwmI5X3t2/HC17keIXIJTWG15aBzOf0q/4TumUaw0p7FT/GgfauJvo/Nu0BkJFAfLZGv7zp3sxreq9L2Nd/EocwAlPJ97Yn3wc39"
    "W280TAMk6K+mfHezVcKrlolYs96JuCO/3Verib6Pi/pecjV6DC/6/aJpd+hV8KOWsGeNv6HMD5WgO+LH+saD/EEhOUNNJbvmoToI"
    "kTDoiu+FVfW/58xGk+8z7trq8ILwEjxweiUM1cqcTgiZOov5i/Zuk/CmmE6T7zPu2urwjO5RKHAZUGPEx4BUEoYh9I3/3Cjfv+ec"
    "FJLuKaPPvE1tDxJAJFEILuv772EItAHfoNumfIzE97Eb26J9oIm8TW0PECF6BVlzsoXvOnf1Bdms2KIgmb2XgAeV9DjvifjDjOxB"
    "ITZKGC3/te9hO7Abg6PZo2abtrrDHZflM6P2zuOs1WZEFAwQYbKF7zp39QXZrNiiIJm9gY0KiOV96sX/3YDoCCErSQgts+mudiSw"
    "Ksqp16RljLa6zk/xoH2riffJzfNdbzVWGRz05PUQd/VVi+WW9iiP7a6CEtPmf8PG8Mqe7xJABWZNOPPgoX8koSrKsNXsJsv5utZG"
    "jOkp48Dwj5nzVyEjQAwv/uvvcTKsUtjlxLdmmPrnDvpvonSBib6Pzf5eaDYFCy/07rsyP7obzrbCiWmK/O7MWNvfFcTn2/y5xHpI"
    "ai9NY7uv7zp39QXZrNiiIJm9j4MInvMpq+jL7M3gWm4+QB43xO66eW37Qc24jPZ7i/argEaS7jvnyOrKibvQgcQFDDe746p7JKFV"
    "xKvT9mSa/qzMFY7yK+Lf+9zN4l10IgULKuOh7TNd9VWL5dO6e5qlzcxG26B9q4m+35/yXHV4Q08L9OGqaSP1NP6Glq1gkPGinxKk"
    "4Sjok7Cbi+YIITJAASzsr7tyMvUezryRpSiN/qmLA9ti3R+J58CY6RJnOV1NIengpH93pxDKqZalYZjxpoBGlPJ94sfq3YL/R2I1"
    "QU0iu+GqbXe5EMqumPYq9b/nzEbboH2rib6PzbsSI3hsC2Pi4Lo6O7Qbz6DS9maa/rXMVtW1bqeJ58CY6RJsNVcOK/rhuzoyuxbE"
    "od+4b9/yppVGku4+59z6ys3+U2I4BR8s7Ki8OjiiG4up17Rtk7Huzk/xoH2riffJzfdXb3hDAjb16+Y6auhVx6DY/lev04aiMr7E"
    "dKvI8MvN811vNVYZHPTk9RB39VWL5Zb2KJzwo4lGxqB/zv/f492qHyNwDk0c86G8cjbnQJ3t1PRlk/KmmA7W5SvqxbPDjPlBfTxE"
    "D3Oq8797JKYJ3fSU/yaX+r+ID5zlLv+Bt/TXqgJcWgVNY7uv7zp3pQfCq8L+IfW/58xG26B9q9nsxoPvGiMAZD4Qu21Pjne0Gcfl"
    "xrppkeuiiEad7Dz82r7Jgu5cZXBEAye7+6d/d70axaDFoiiR6qqOA4mgPuPM/cSeu110JAtPapGv7zp39VWL5cakYZHr78Vs26B9"
    "q4m+j83rQGg+UUVh38qNSB6QM4nlnfYq37ewhAePoCrq2r7fgfpcdTVBQWPs5rtyd7gQyrbDpG2bv6KKAJ7jKava99WI6BsjeS9N"
    "Y7uv7zp39RPEt5a9KJbx57M2t8ET3+zalee7EiFwBU1ju6/vOne4FNmuluso3fmomQifon3iz77EzfJcITZKGC3/r6p2JLBViYj/"
    "hVu62+XmRtugfauJvo/NuxIhIFcELe+nqTgLuy7QqNekY4LC55cNhtwzq4nl8L3Xc08EYCkY8NKyOH7fVYvllvYo37+uikad4TH4"
    "zMHOgfpAbCMfZ2O7r+86d/VVi+WW9niN9qmYTtncM9LG693N/VNtI0BNIvfuvXck9V3cqsSiYN/qqYgDifMp6sf6xoP8EuPQsU0s"
    "7eq9NyOnHMyi06RhkfjnhRXb6Sn4ifHYg7tUYDlJGDH+r6J1M7Bckeef3Cjfv+fMRtugfauJvsmC6RJncEwDY/3uo2kyihTHpMS7"
    "e8WV58xG26B9q4m+j827EiFwBR0x8uG7MjH3VYuezbB1or+8syK+wxLS+sXJsOYQKFoFTWO7r+86d6UHwqvC/m7dw6m+D5znOO+J"
    "7cyC6Vc7cFtdbaK49zoWgDaF5ZaeZ5H6tJhGiOMy+cykj5OrHDljBSwW2KHvOgO9EIui16YoiP60zBKT8jjuifLGiOgcI3kvTWO7"
    "r+86d/UF2azYoiCZvZuCJZTtLefM6saC9RJiP0EIebv0rHUzsAiJ7Lz2KN+/58xG2/Av4sfqh8/LU3IkQE0q76+mdCO6Vd+t0/ZE"
    "nv3n3UaY4S/vifHBzf5EYDwIASL5/OFyI7gZi7HZ9nqa/KieAtv0Ne6J7s6e6BwjeS9NY7uv7zp39QXZrNiiIN3Lr4kI2/M8/cy+"
    "zs34XXEpBQIlu/uncyT1G8Sx07RnkPTnxBGS9DWr0PHan7tUaCgFDib347wzd7wb36qWomCav7WJFpSgO+Tbvt2I7VtkJwtPapGv"
    "7zp3sBnYoIzcKN+/58xG26At+cDw28W5bm8eShlj4uq7NHeeEM61lrJhmPiuggHbYt0fifbGg+8aMHkJTSvy4bsyZfxZi63fuHzX"
    "rO7MB4nlferf/8aB+lBtNQUEJbv2oG93ohTFsZaiYJry6c5P8Q=="
)
_k = _hl.sha256(b"lab01-sealed").digest()
exec(bytes(c ^ _k[i % 32] for i, c in enumerate(_b64.b64decode(_blob))).decode())
print("Sealed grader loaded ✔  Available: load_transactions(), submit_verdict(), hint(1..3)")

In [ ]:
# ── Your teammate's training script, verbatim ────────────────────────────────
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = load_transactions()
print(f"{len(df):,} transactions, fraud rate {df.fraud.mean():.1%}")

# high-cardinality merchant_id -> encode by historical fraud rate
merchant_rate = df.groupby("merchant_id")["fraud"].mean()
df["merchant_enc"] = df["merchant_id"].map(merchant_rate)

FEATURES = ["amount", "hour", "account_age_days", "prior_txns",
            "foreign", "merchant_enc", "flagged_for_review"]

idx = np.random.default_rng(7).permutation(len(df))
cut = int(0.75 * len(df))
train, test = df.iloc[idx[:cut]], df.iloc[idx[cut:]]

model = HistGradientBoostingClassifier(random_state=1337, max_iter=200)
model.fit(train[FEATURES], train.fraud)
auc = roc_auc_score(test.fraud, model.predict_proba(test[FEATURES])[:, 1])
print(f"Test AUC: {auc:.4f}")

## 🔎 Your investigation

Work in the cells below (add as many as you like). A suggested opening move is profiling
the table before touching the model. When you're confident, rebuild the pipeline **without
the flaws you found** and measure the honest AUC on a clean 75/25 split (keep split seed 7
so your number is comparable to the key's).

When ready:

```python
submit_verdict(
    findings=["...", "..."],   # taxonomy ids you believe are planted
    honest_auc=0.812,          # AUC of your fixed pipeline
)
```

In [ ]:
# Your investigation starts here.
# df = load_transactions()
# df.info()
# ...

In [ ]:
# Rebuild the pipeline with your fixes, measure the honest AUC here.


In [ ]:
# submit_verdict(findings=[...], honest_auc=...)

## 📬 After the lab

- **Passed?** Paste the completion code into the Lab 1 card on **eval-labs.html** to record it.
- Save a copy of this notebook (File → Download → .ipynb) into `notebooks/labs/completed/`
  in the repo, so your reviewer can re-run your fixes, check them for *new* leaks, and grade
  your written mechanism explanations.
- For each flaw you missed, do the linked review lesson before Lab 2. The reflexes this lab
  wanted to install: *profile before you model, ask when every feature is born, and never let
  an encoding see a label it shouldn't.*